In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../../../../../Data/HousingData.csv")
df

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,NaN,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0.0,0.573,6.593,69.1,2.4786,1,273,21.0,391.99,NaN,22.4
502,0.04527,0.0,11.93,0.0,0.573,6.120,76.7,2.2875,1,273,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0.0,0.573,6.976,91.0,2.1675,1,273,21.0,396.90,5.64,23.9
504,0.10959,0.0,11.93,0.0,0.573,6.794,89.3,2.3889,1,273,21.0,393.45,6.48,22.0


In [3]:
df.dropna(inplace=True)

In [4]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [9]:
from sklearn.metrics import r2_score

def modelR2Score(model):
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    return {
        "R2 Score:": r2_score(y_test, y_pred),
    }

# 1. Classification with different algo

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

estimator_list = [
    ("lr",LinearRegression()),
    ("dtr",DecisionTreeRegressor()),
    ("svr", SVR())
]

### validate every model

In [11]:
from sklearn.model_selection import cross_val_score

for model in estimator_list:
    score = np.mean(cross_val_score(model[1], X,y, scoring='r2', cv=10))
    print(model[0]," : ",round(score,3))

lr  :  0.293
dtr  :  0.185
svr  :  -0.37


# 1.1 Voting Regressor

In [12]:
from sklearn.ensemble import VotingRegressor

vote_reg = VotingRegressor(estimators=estimator_list)
print("Cross Val Score: ",np.mean(cross_val_score(vote_reg, X,y, cv=10, scoring='r2')))
print(modelR2Score(vote_reg))

Cross Val Score:  0.3893026932943517
{'R2 Score:': 0.5923102784589547}


# 1.3 Weighted Voting

In [13]:
def weightedVoting(max_weight):
    maxAcc, ithm, jthm, kthm = 0, 0, 0, 0
    for i in range(1, max_weight+1):
        for j in range(1, max_weight+1):
            for k in range(1, max_weight+1):
                weightedVote_clf = VotingRegressor(estimators=estimator_list, weights=[i,j,k])
                score = np.mean(cross_val_score(weightedVote_clf, X, y, scoring='r2', cv=5))
                if score > maxAcc:
                    maxAcc = score
                    ithm = i 
                    jthm = j
                    kthm = k
    return maxAcc, ithm, jthm, kthm

In [14]:
weightedVoting(4)

(np.float64(0.5468744508882244), 4, 4, 1)

In [15]:
weighted_vote_clf = VotingRegressor(estimators=estimator_list, weights=[4,4,1])
print("Cross Val Score: ",np.mean(cross_val_score(weighted_vote_clf, X,y, cv=10, scoring='r2')))
print(modelR2Score(weighted_vote_clf))

Cross Val Score:  0.45688868888917017
{'R2 Score:': 0.6773740288844896}


# 2. Classification with same alog

In [16]:
same_estimator_list = [
    ("svc1",DecisionTreeRegressor(max_depth=1)),
    ("svc2",DecisionTreeRegressor(max_depth=3)),
    ("svc3",DecisionTreeRegressor(max_depth=5)),
    ("svc4",DecisionTreeRegressor(max_depth=7)),
    ("svc5",DecisionTreeRegressor(max_depth=None))
]

In [17]:
for model in same_estimator_list:
    score = np.mean(cross_val_score(model[1], X,y, scoring='r2', cv=10))
    print(model[0]," : ",round(score,3))

svc1  :  -0.913
svc2  :  -0.038
svc3  :  0.262
svc4  :  0.181
svc5  :  0.193


In [19]:
sameAlgo_vote_reg = VotingRegressor(estimators=same_estimator_list)
print("Cross Val Score: ",np.mean(cross_val_score(sameAlgo_vote_reg, X,y, cv=10, scoring='r2')))
print(modelR2Score(sameAlgo_vote_reg))

Cross Val Score:  0.2514485723464036
{'R2 Score:': 0.6680873738850108}
